In [95]:
# begin redevelop from scratch for synthesis plugin
# clean up the LP and result objects
# start with backend interface so avoid all the mess from the original version
# https://docs.quantum.ibm.com/api/qiskit/transpiler_synthesis_plugins#unitary-synthesis-plugins
import numpy as np
import qiskit
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit.circuit.library import RZXGate, UnitaryGate, XXPlusYYGate
from qiskit.converters import circuit_to_dag
from qiskit.transpiler import InstructionProperties, Target

print(qiskit.__version__)

2.0.1


In [96]:
# create some mock hetereogeneous ISA backend
target = Target()

target.add_instruction(
    RZXGate(np.pi / 2),
    {
        (0, 1): InstructionProperties(
            duration=100,
            error=0.01,
        )
    },
    name="cx",
)
target.add_instruction(
    RZXGate(np.pi / 4),
    {
        (0, 1): InstructionProperties(
            duration=50,
            error=0.05,
        )
    },
    name="scx",
)

target.add_instruction(
    XXPlusYYGate(np.pi),
    {
        (0, 1): InstructionProperties(
            duration=200,
            error=0.02,
        )
    },
    name="iswap",
)
target.add_instruction(
    XXPlusYYGate(np.pi / 2),
    {
        (0, 1): InstructionProperties(
            duration=100,
            error=0.01,
        )
    },
    name="siswap",
)

In [ ]:
# decomposition logic goes here
def generate_dag_circuit_from_matrix(unitary, *args):  # noqa: D103
    qc = QuantumCircuit(2)
    qc.append(UnitaryGate(unitary), [0, 1])
    return circuit_to_dag(qc)

In [ ]:
from qiskit.transpiler.passes.synthesis import plugin


class GULPSynthesis(plugin.UnitarySynthesisPlugin):
    """GULP synthesis plugin."""

    @property
    def min_qubits(self):  # noqa: D102
        return 2

    @property
    def max_qubits(self):  # noqa: D102
        return 2

    @property
    def supports_target(self):  # noqa: D102
        return True

    ######################################
    # NOTE, I avoid using basis_gates and gate_errors to focus on implementation
    # that uses the target object
    @property
    def supports_basis_gates(self):  # noqa: D102
        return False

    @property
    def supports_gate_errors(self):  # noqa: D102
        return False

    # NOTE, using this one seems better than supports_gate_errors
    # but leave False for now
    @property
    def supports_gate_errors_by_qubit(self):  # noqa: D102
        return False

    @property
    def supports_coupling_map(self):  # noqa: D102
        return False

    @property
    def supports_natural_direction(self):  # noqa: D102
        return False

    @property
    def supports_pulse_optimize(self):  # noqa: D102
        return False

    @property
    def supports_gate_lengths(self):  # noqa: D102
        return False

    @property
    def supports_gate_lengths_by_qubit(self):  # noqa: D102
        return False

    @property
    def supported_bases(self):  # noqa: D102
        return None

    def run(self, unitary, **options):  # noqa: D102
        # configure settings
        target = options.get("target")
        if target is None:
            raise ValueError("Target must be provided in options.")

        # XXX I'll just assume we are only working with 2 qubit targets for now
        isa_tuple = []
        for idx, instruction in enumerate(target.operations_for_qargs((0, 1))):
            error = target.instruction_properties(idx).error
            isa_tuple.append((instruction, error))

        # call the synthesis function
        dag_circuit = generate_dag_circuit_from_matrix(
            unitary, basis_gates, gate_errors
        )
        return dag_circuit

In [ ]:
target.operations_for_qargs((0, 1))[0]

Instruction(name='xx_plus_yy', num_qubits=2, num_clbits=0, params=[3.141592653589793, 0])

In [105]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

circuit = QuantumCircuit(2)
circuit.cx(0, 1)

unitary = Operator(circuit).data

plugin = GULPSynthesis()
out = plugin.run(unitary, target=target)
out

AttributeError: 'Target' object has no attribute 'basis_gates'